# 🚀 Kaggle Nomic Text Embedding & PDF Parser Server
This notebook loads the **nomic-ai/nomic-embed-text-v1.5** model and **Docling** to serve as a high-performance, GPU-accelerated PDF parsing and embeddings backend. It exposes the API to the public internet using a secure `pyngrok` tunnel.

### ⚙️ Instructions
1. Turn on the **GPU Accelerator** in the notebook settings (select **GPU T4 x2** or **GPU T4 x1**).
2. Add your **Ngrok Auth Token** to Kaggle Secrets:
   - Go to **Add-ons -> Secrets** in the top menu.
   - Add a secret named `NGROK_AUTH_TOKEN` with your token as the value.
   - Make sure the checkbox next to the secret is checked to share it with the notebook.
3. Run all cells in order.

## 📦 Step 1: Install Dependencies

In [ ]:
!pip install -q \
    transformers \
    accelerate \
    torch \
    fastapi \
    uvicorn \
    pyngrok \
    nest_asyncio \
    pydantic \
    einops \
    docling \
    python-multipart \
    pandas \
    tabulate

## 🔑 Step 2: Load Secrets and Import Libraries

In [ ]:
import os
import tempfile
import uuid
import torch
import nest_asyncio
import logging
from fastapi import FastAPI, UploadFile, Form, File, BackgroundTasks
from pydantic import BaseModel
from pyngrok import ngrok
from transformers import AutoTokenizer, AutoModel
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions, RapidOcrOptions
from docling.datamodel.base_models import InputFormat
from docling.chunking import HybridChunker

# Configure system-wide logging inside Kaggle notebook to print Docling progress logs
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S"
)
logging.getLogger("docling").setLevel(logging.DEBUG)
logging.getLogger("docling.pipeline.base_pipeline").setLevel(logging.DEBUG)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)

MODEL_NAME = "nomic-ai/nomic-embed-text-v1.5"
PORT = 8000

# Fetch secret token from Kaggle user secrets
NGROK_AUTH_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_AUTH_TOKEN")
    print("✅ Loaded NGROK_AUTH_TOKEN from Kaggle User Secrets.")
except Exception as e:
    print("❌ Failed to load secret from Kaggle User Secrets. Checking environment variables...")
    NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN")

if not NGROK_AUTH_TOKEN:
    print("⚠️ WARNING: NGROK_AUTH_TOKEN is not set. Please add it to Add-ons -> Secrets.")

## 🤖 Step 3: Load Nomic Embedding Model & Docling

In [ ]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

print("Loading model...")
model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = model.to("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
print(f"✅ Embedding model loaded successfully onto: {model.device}")

print("Configuring Docling PDF Converter with GPU...")
pipeline_options = PdfPipelineOptions()
pipeline_options.accelerator_options.device = "cuda" if torch.cuda.is_available() else "cpu"

# Explicitly configure RapidOCR to use the Torch backend so it utilizes the CUDA GPU
pipeline_options.ocr_options = RapidOcrOptions(backend="torch")

doc_converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)
chunker = HybridChunker(max_tokens=512)
print("✅ Docling pipeline loaded successfully.")

## 🛠️ Step 4: Define API Schemas and Router

In [ ]:
app = FastAPI(title="Kaggle Nomic Embedding & Docling API")

class EmbedRequest(BaseModel):
    texts: list[str]

class EmbedResponse(BaseModel):
    success: bool
    embeddings: list[list[float]]
    error_message: str

tasks_store = {}

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

def get_embeddings_internal(texts: list[str]) -> list[list[float]]:
    import torch.nn.functional as F
    
    encoded_input = tokenizer(texts, padding=True, truncation=True, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        model_output = model(**encoded_input)
        
    embeddings = mean_pooling(model_output, encoded_input['attention_mask'])
    embeddings = F.normalize(embeddings, p=2, dim=1)
    
    return embeddings.tolist()

def process_pdf_in_background(task_id: str, tmp_path: str, company: str, year: int):
    try:
        print(f"[{task_id}] Starting background parsing on GPU...")
        result = doc_converter.convert(tmp_path)
        doc = result.document

        chunk_list = list(chunker.chunk(dl_doc=doc))

        chunks = []
        for chunk in chunk_list:
            pages = sorted({
                prov.page_no 
                for item in chunk.meta.doc_items 
                for prov in item.prov 
                if hasattr(prov, "page_no")
            })
            page_number = pages[0] if pages else 1
            
            section_path = " > ".join(chunk.meta.headings) if chunk.meta.headings else "General"
            context_text = chunker.contextualize(chunk)
            
            chunks.append({
                "text": context_text,
                "page_number": page_number,
                "section": section_path
            })

            if not chunks:
                tasks_store[task_id] = {"status": "completed", "chunks": []}
                return

        print(f"[{task_id}] Generating embeddings for {len(chunks)} chunks...")
        texts_to_embed = [
            f"search_document: [Company: {company} | Year: {year}] {c['text']}"
            for c in chunks
        ]
        
        batch_size = 32
        all_embeddings = []
        for i in range(0, len(texts_to_embed), batch_size):
            batch = texts_to_embed[i:i+batch_size]
            all_embeddings.extend(get_embeddings_internal(batch))

        output_chunks = []
        for chunk, embedding in zip(chunks, all_embeddings):
            output_chunks.append({
                "text": chunk["text"],
                "page_number": chunk["page_number"],
                "section": chunk["section"],
                "embedding": embedding
            })

        tasks_store[task_id] = {
            "status": "completed",
            "chunks": output_chunks
        }
        print(f"[{task_id}] Background parsing completed successfully.")
    except Exception as e:
        import traceback
        err_msg = traceback.format_exc()
        print(f"[{task_id}] Error in background task:\n{err_msg}")
        tasks_store[task_id] = {
            "status": "failed",
            "error_message": str(e)
        }
    finally:
        import os
        if os.path.exists(tmp_path):
            os.remove(tmp_path)

@app.get("/")
def root():
    return {
        "status": "running",
        "model": MODEL_NAME
    }

@app.post("/embed")
def embed(req: EmbedRequest):
    try:
        embeddings = get_embeddings_internal(req.texts)
        return EmbedResponse(
            success=True,
            embeddings=embeddings,
            error_message=""
        )
    except Exception as e:
        return EmbedResponse(
            success=False,
            embeddings=[],
            error_message=str(e)
        )

@app.post("/parse-pdf-async")
async def parse_pdf_async(
    background_tasks: BackgroundTasks,
    file: UploadFile = File(...),
    company: str = Form(...),
    year: int = Form(...)
):
    try:
        task_id = str(uuid.uuid4())
        tasks_store[task_id] = {"status": "processing"}
        
        with tempfile.NamedTemporaryFile(suffix=".pdf", delete=False) as tmp:
            content = await file.read()
            tmp.write(content)
            tmp_path = tmp.name

        background_tasks.add_task(
            process_pdf_in_background,
            task_id,
            tmp_path,
            company,
            year
        )
        
        return {
            "success": True,
            "task_id": task_id
        }
    except Exception as e:
        return {
            "success": False,
            "task_id": "",
            "error_message": str(e)
        }

@app.get("/parse-status/{task_id}")
def get_parse_status(task_id: str):
    status = tasks_store.get(task_id)
    if not status:
        return {"status": "not_found"}
    return status

## 🚀 Step 5: Start Public Tunnel and FastAPI Server

In [ ]:
# Apply nesting patch to allow uvicorn inside Jupyter event loop
nest_asyncio.apply()

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    public_url = ngrok.connect(PORT)
    print("\n" + "="*80)
    print(f"🚀 SERVER PUBLIC URL: {public_url}")
    print("Copy the URL above and paste it into your Backed/.env as EMBEDDING_SERVER_URL")
    print("="*80 + "\n")
else:
    print("\n❌ Unable to set up ngrok tunnel without an authentication token.")

# Launch server directly in the notebook's active event loop
import uvicorn
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, loop="asyncio")
server = uvicorn.Server(config)
await server.serve()